# Generación del conjunto de datos de entrenamiento (Churn)

El objetivo de esta libreta es construir el conjunto de datos de entrenamiento para un modelo de predicción de churn en Telco.

Se combinan:
- gold_churn_spine
- gold_customer_profile
- gold_customer_aggregations

El resultado se guarda como:
👉 gold_churn_training_dataset

In [ ]:
%pip install databricks-feature-engineering>=0.13.0
dbutils.library.restartPython()

In [ ]:
from databricks.feature_engineering import FeatureEngineeringClient, FeatureLookup
from datetime import datetime, timezone
from pyspark.sql.functions import col, count, max as spark_max, round, when


In [ ]:
catalog = "workspace"
database = "telco_churn"

gold_spine_table = f"{catalog}.{database}.gold_churn_spine"
gold_customer_profile_table = f"{catalog}.{database}.gold_customer_profile"
gold_customer_aggregations_table = f"{catalog}.{database}.gold_customer_aggregations"

gold_training_dataset_table = f"{catalog}.{database}.gold_churn_training_dataset"

fe = FeatureEngineeringClient()

In [ ]:
spine_df = spark.table(gold_spine_table)

print(f"Rows: {spine_df.count():,}")
print(f"Columns: {len(spine_df.columns)}")

spine_df.printSchema()

In [ ]:
entity_key = "customer_id"
timestamp_key = "usage_event_time"

profile_feature_names = [
    "age","gender","contract_type","region","region_type",
    "tariff_plan","monthly_fee","num_lines","device_type",
    "acquisition_channel","payment_method","has_tv_bundle",
    "has_fiber","has_roaming","paperless_billing","autopay",
    "nps_score_at_start","is_active","age_group","contract_risk_group",
    "signup_date"
]

profile_lookup = FeatureLookup(
    table_name=gold_customer_profile_table,
    feature_names=profile_feature_names,
    lookup_key=entity_key,
    timestamp_lookup_key=timestamp_key
)

aggregation_feature_names = [
    "data_consumed_gb","call_minutes","bill_amount",
    "days_payment_late","nps_score","coverage_score",
    "bill_vs_data_ratio"
]

aggregations_lookup = FeatureLookup(
    table_name=gold_customer_aggregations_table,
    feature_names=aggregation_feature_names,
    lookup_key=entity_key,
    timestamp_lookup_key=timestamp_key
)

feature_lookups = [profile_lookup, aggregations_lookup]

In [ ]:
label = "label_will_churn"
exclude_columns = ["churn_date", "label_available_date"]

training_dataset = fe.create_training_set(
    df = spine_df,
    feature_lookups = feature_lookups,
    label = label,
    exclude_columns = exclude_columns
)

print("Training dataset logical plan created.")
print(f"Label column: {label}")
print(f"Excluded columns: {exclude_columns}")


In [ ]:
training_df = training_dataset.load_df()

print(f"Rows: {training_df.count():,}")
print(f"Columns: {len(training_df.columns)}")

training_df.printSchema()

In [ ]:
training_df.limit(5).toPandas()

In [ ]:
non_feature_columns = {
    "customer_id",
    "usage_event_time",
    "year_month",
    "window_end",
    "label_will_churn",
    "churn_date",
    "label_available_date",
    "__START_AT",
    "__END_AT",
}
feature_columns = [
    c for c in training_df.columns
    if c not in non_feature_columns
]

nulls = training_df.select([
    count(when(col(c).isNull(), c)).alias(c) for c in feature_columns
]).collect()[0]

for c in feature_columns:
    print(f"{c}: {nulls[c]:,}")


In [ ]:
print("Spine:", spine_df.count())
print("Training:", training_df.count())

In [ ]:
total = training_df.count()

balance = (
    training_df.groupBy("label_will_churn")
    .count()
    .withColumn("pct", round(col("count") / total * 100, 2))
    .collect()
)

for r in balance:
    print(r)

In [ ]:
clean_training_df = training_df.filter("label_will_churn IS NOT NULL")

print(f"Final rows: {clean_training_df.count():,}")

In [ ]:
def _table_exists(table_name):
    try:
        spark.table(table_name).limit(1).collect()
        return True
    except Exception:
        return False


def _get_current_training_metadata(table_name):
    if not _table_exists(table_name):
        return {
            "semantic_version": None,
            "data_max_date": None,
            "data_previous_max_date": None,
        }

    properties_df = spark.sql(f"SHOW TBLPROPERTIES {table_name}")

    def _property_value(key):
        row = properties_df.filter(f"key = '{key}'").first()
        return row["value"] if row else None

    semantic_version = _property_value("ml.delta_semantic_version")
    return {
        "semantic_version": int(semantic_version) if semantic_version else None,
        "data_max_date": _property_value("ml.data_max_date"),
        "data_previous_max_date": _property_value("ml.data_previous_max_date"),
    }


def _format_date(value):
    if value is None:
        return ""
    if isinstance(value, str):
        return value[:10]
    return value.strftime("%Y-%m-%d")


clean_training_count = clean_training_df.count()
current_training_metadata = _get_current_training_metadata(gold_training_dataset_table)
previous_table_max_date = current_training_metadata["data_max_date"]

(
    clean_training_df
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .option("delta.enableChangeDataFeed", "true")
    .saveAsTable(gold_training_dataset_table)
)

delta_physical_version = int(
    spark.sql(f"DESCRIBE HISTORY {gold_training_dataset_table}")
         .select("version")
         .first()[0]
)

data_max_date = _format_date(
    clean_training_df
    .agg(spark_max(col(timestamp_key)).alias("max_date"))
    .collect()[0]["max_date"]
)
if current_training_metadata["semantic_version"] is None:
    delta_semantic_version = 0
    data_previous_max_date = ""
elif previous_table_max_date and data_max_date > previous_table_max_date:
    delta_semantic_version = current_training_metadata["semantic_version"] + 1
    data_previous_max_date = previous_table_max_date
else:
    delta_semantic_version = current_training_metadata["semantic_version"]
    data_previous_max_date = current_training_metadata["data_previous_max_date"] or ""

generated_at = datetime.now(timezone.utc).isoformat()

spark.sql(f"""
    ALTER TABLE {gold_training_dataset_table}
    SET TAGS (
        'delta_semantic_version' = '{delta_semantic_version}',
        'delta_physical_version' = '{delta_physical_version}',
        'data_max_date' = '{data_max_date}',
        'data_previous_max_date' = '{data_previous_max_date}',
        'spine_table' = '{gold_spine_table}',
        'feature_table_1' = '{gold_customer_profile_table}',
        'feature_table_2' = '{gold_customer_aggregations_table}',
        'label_col' = '{label}',
        'num_rows' = '{clean_training_count}',
        'num_features' = '{len(feature_columns)}',
        'generated_at' = '{generated_at}'
    )
""")

spark.sql(f"""
    ALTER TABLE {gold_training_dataset_table}
    SET TBLPROPERTIES (
        'ml.delta_semantic_version' = '{delta_semantic_version}',
        'ml.delta_physical_version' = '{delta_physical_version}',
        'ml.data_max_date' = '{data_max_date}',
        'ml.data_previous_max_date' = '{data_previous_max_date}',
        'ml.spine_table' = '{gold_spine_table}',
        'ml.feature_table_1' = '{gold_customer_profile_table}',
        'ml.feature_table_2' = '{gold_customer_aggregations_table}',
        'ml.label_col' = '{label}',
        'ml.num_rows' = '{clean_training_count}',
        'ml.num_features' = '{len(feature_columns)}',
        'ml.generated_at' = '{generated_at}'
    )
""")

table_description = f"""Static training dataset for the Telco churn model.
It is generated by a point-in-time join between `{gold_spine_table}`,
`{gold_customer_profile_table}` and `{gold_customer_aggregations_table}`,
filtered to supervised rows and versioned for MLflow traceability."""
spark.sql(f"COMMENT ON TABLE {gold_training_dataset_table} IS '{table_description}'")

print(f"Training dataset saved to: {gold_training_dataset_table}")
print(f"Semantic version: {delta_semantic_version}")
print(f"Physical version: {delta_physical_version}")
print(f"Data maximum date: {data_max_date}")
print(f"Data previous maximum date: {data_previous_max_date or 'none'}")
print(f"Number of rows: {clean_training_count:,}")
print(f"Number of features: {len(feature_columns)}")
print(f"Generated at: {generated_at}")

df_past = (
    spark.read
         .format("delta")
         .option("versionAsOf", delta_physical_version)
         .table(gold_training_dataset_table)
)
print(f"Rows loaded via time travel: {df_past.count():,}")
